In [1]:
# ─────────────────────────────────────────────
# 환경 준비 — 라이브러리 불러오기 + 한글 폰트 + 시드 고정
# ─────────────────────────────────────────────
# 필요 시 아래 주석을 해제해 설치하세요.
# !pip install numpy pandas matplotlib seaborn pyarrow -q

import os
import platform
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")   # 학습 중 경고 메시지를 잠시 숨깁니다.

# 재현성: 같은 난수를 항상 같게 만들어 결과가 매번 동일하도록 고정합니다.
np.random.seed(42)

# 한글 폰트 설정 (그래프 안 글자가 깨지지 않도록 운영체제별로 분기)
system = platform.system()
if system == "Darwin":          # macOS
    plt.rcParams["font.family"] = "AppleGothic"
elif system == "Windows":       # Windows
    plt.rcParams["font.family"] = "Malgun Gothic"
else:                            # Linux 등
    plt.rcParams["font.family"] = "DejaVu Sans"

plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.figsize"] = (10, 5)
sns.set_style("whitegrid")

# 결과 저장용 임시 폴더 (이 노트북 옆에 'd009_outputs/' 가 만들어집니다)
OUT_DIR = Path("d009_outputs")
OUT_DIR.mkdir(exist_ok=True)

print("준비 완료! 라이브러리 버전을 확인합니다.")
print("numpy :", np.__version__)
print("pandas:", pd.__version__)
print("저장 폴더:", OUT_DIR.resolve())

준비 완료! 라이브러리 버전을 확인합니다.
numpy : 2.5.4
pandas: 3.0.6
저장 폴더: C:\Users\chaee\OneDrive\Desktop\DI-2\data-wrangling\D009\d009_outputs


In [2]:
# ─────────────────────────────────────────────
# 종합 프로젝트용 모두마켓 데이터 생성 — 이 셀 하나로 오늘 쓸 데이터가 모두 준비됩니다.
# (실제 현장처럼 '오염'을 일부러 심어 둡니다: 결측·이상치·표기 혼재·날짜 포맷 혼재·중복·타입 불일치)
# ─────────────────────────────────────────────
np.random.seed(42)

# 1) 고객(customers)
n_customers = 500
customers = pd.DataFrame({
    "customer_id": [f"C{str(i).zfill(4)}" for i in range(1, n_customers + 1)],
    "age": np.random.normal(36, 10, n_customers).round().astype(int),
    "gender": np.random.choice(["M", "F"], n_customers),
    "region": np.random.choice(["서울", "경기", "부산", "인천", "대구"], n_customers),
    "membership": np.random.choice(["basic", "premium", "vip"], n_customers, p=[0.6, 0.3, 0.1]),
    "signup_date": pd.to_datetime("2024-01-01") + pd.to_timedelta(
        np.random.randint(0, 700, n_customers), unit="D"
    ),
})
# 오염 심기: 나이 이상치, 결측, 지역/멤버십 표기 혼재
customers.loc[[5, 6], "age"] = 999
customers.loc[10, "age"] = -3
customers.loc[[20, 21, 22, 23], "gender"] = np.nan
customers.loc[30, "region"] = " 서울 "
customers.loc[31, "region"] = "Seoul"
customers.loc[40, "membership"] = "VIP"
customers.loc[41, "membership"] = " premium"

# 2) 상품(products)
categories = ["패션", "뷰티", "식품", "가전", "도서"]
n_products = 60
products = pd.DataFrame({
    "product_id": [f"P{str(i).zfill(3)}" for i in range(1, n_products + 1)],
    "category": np.random.choice(categories, n_products),
    "price": np.random.choice([9900, 19900, 29900, 49900, 89900, 129900, 199900], n_products),
})

# 3) 주문(orders)
n_orders = 5000
order_customer = np.random.choice(customers["customer_id"], n_orders)
order_product = np.random.choice(products["product_id"], n_orders)
price_map = products.set_index("product_id")["price"]
quantity = np.random.choice([1, 1, 1, 2, 2, 3], n_orders)
amount = price_map.loc[order_product].values * quantity

orders = pd.DataFrame({
    "order_id": [f"O{str(i).zfill(5)}" for i in range(1, n_orders + 1)],
    "customer_id": order_customer,
    "product_id": order_product,
    "quantity": quantity,
    "amount": amount.astype(float),
    "channel": np.random.choice(["web", "app", "app ", "APP"], n_orders, p=[0.45, 0.45, 0.05, 0.05]),
})

# 날짜 포맷 혼재(문자열로 저장 — 일부러 통일하지 않음)
base_dates = pd.to_datetime("2025-01-01") + pd.to_timedelta(np.random.randint(0, 180, n_orders), unit="D")
date_strings = []
for i, d in enumerate(base_dates):
    if i % 3 == 0:
        date_strings.append(d.strftime("%Y-%m-%d"))
    elif i % 3 == 1:
        date_strings.append(d.strftime("%Y/%m/%d"))
    else:
        date_strings.append(d.strftime("%Y%m%d"))
orders["order_date"] = date_strings

# 오염 심기: 금액 결측, 수량 이상치, 중복 행, 음수 금액(환불로 보이는 값)
orders.loc[np.random.choice(n_orders, 80, replace=False), "amount"] = np.nan
orders.loc[7, "quantity"] = 100
orders.loc[8, "amount"] = -50000.0   # 환불 또는 입력 실수 (판단 필요)
orders = pd.concat([orders, orders.iloc[[0, 1, 2]]], ignore_index=True)  # 중복 3건

# 4) 웹 로그(web_logs) — 텍스트 한 줄로 들어옴(D+005 식 파싱 대상)
n_logs = 1200
log_dates = pd.to_datetime("2025-04-01") + pd.to_timedelta(np.random.randint(0, 60, n_logs), unit="D")
ips = [f"{np.random.randint(1,254)}.{np.random.randint(0,254)}.{np.random.randint(0,254)}.{np.random.randint(0,254)}" for _ in range(n_logs)]
paths = np.random.choice(["/home", "/product/123", "/cart", "/checkout", "/search?q=shoes"], n_logs)
web_logs = pd.DataFrame({
    "raw_log": [f"{d.strftime('%Y-%m-%d %H:%M:%S')} - {ip} - GET {p}" for d, ip, p in zip(log_dates, ips, paths)]
})

# CSV로 임시 저장(인제스트 단계에서 읽어옴) — 분석가가 "파일을 받았다" 상황 재현
RAW_DIR = OUT_DIR / "raw"
RAW_DIR.mkdir(exist_ok=True)
customers.to_csv(RAW_DIR / "customers.csv", index=False)
products.to_csv(RAW_DIR / "products.csv", index=False)
orders.to_csv(RAW_DIR / "orders.csv", index=False)
web_logs.to_csv(RAW_DIR / "web_logs.csv", index=False)

print("종합 프로젝트 데이터 생성 및 CSV 저장 완료")
print(f"  customers: {customers.shape}")
print(f"  products : {products.shape}")
print(f"  orders   : {orders.shape}  (중복·결측·이상치 포함)")
print(f"  web_logs : {web_logs.shape}")
print(f"\n저장 경로: {RAW_DIR.resolve()}")

종합 프로젝트 데이터 생성 및 CSV 저장 완료
  customers: (500, 6)
  products : (60, 3)
  orders   : (5003, 7)  (중복·결측·이상치 포함)
  web_logs : (1200, 1)

저장 경로: C:\Users\chaee\OneDrive\Desktop\DI-2\data-wrangling\D009\d009_outputs\raw


In [3]:
# 예제: 품질 리포트 함수 v1 — 결측·중복·타입을 한 번에 보여주는 진단기
def quality_report(df: pd.DataFrame, name: str = "df") -> pd.DataFrame:
    '''데이터프레임의 품질을 컬럼별로 진단해 한 표로 돌려줍니다.'''
    n_rows = len(df)
    report = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(2),
        "n_unique": df.nunique(dropna=True),
        "sample": [df[col].dropna().iloc[0] if df[col].notna().any() else None for col in df.columns],
    })
    print(f"[품질 리포트] {name}")
    print(f"  행 수: {n_rows:,}  /  열 수: {len(df.columns)}")
    print(f"  메모리: {df.memory_usage(deep=True).sum() / 1024:.1f} KB")
    print(f"  완전 중복 행: {df.duplicated().sum()}건")
    return report


# orders에 적용
qr_orders = quality_report(orders_csv, "orders_csv")
qr_orders

NameError: name 'orders_csv' is not defined